In [1]:
import os
import sys



In [2]:
pwd

'/mnt/d/resume_projects/flight_fare_prediction/research'

In [3]:

os.chdir("../")
%pwd

'/mnt/d/resume_projects/flight_fare_prediction'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    feature_store_file_name: Path
    training_file_name: Path
    testing_file_name: Path
    train_test_split_ratio: float
    collection_name: str
    database_name: str

In [5]:
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.utils.common import read_yaml, create_directories , create_directory
from src.flight_price_prediction.entity.config_entity import DataIngestionConfig


In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema =read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config =self.config.data_ingestion
        params = self.params.data_ingestion
        create_directory(config.root_dir)

        data_ingestion_config = DataIngestionConfig(
            root_dir = Path(config.root_dir),
            feature_store_file_name = Path(config.feature_store_file_name),
            training_file_name = Path(config.train_file_name),
            testing_file_name = Path(config.testing_file_name),
            train_test_split_ratio = params.train_test_split_ratio,
            collection_name = config.collection_name,
            database_name=config.database_name
        )

        return data_ingestion_config



In [7]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pymongo
from typing import List
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.entity.config_entity import DataIngestionConfig
from src.flight_price_prediction.entity.artifact_entity import DataIngestionArtifact
from src.flight_price_prediction.utils.common import *
from dotenv import load_dotenv
load_dotenv()

MONGO_DB_URL = os.getenv("MONGO_DB_URL")

class DataIngestion:
    def __init__(self, data_ingestion_config: DataIngestionConfig):
        """ 
         Initiates the DataIngestion component.
         Ensure your MONGO_DB_URL is set in your environment variables
         
         """
        try:
            self.data_ingestion_config = data_ingestion_config
            self.mongo_db_url = os.getenv("MONGO_DB_URL")
            self.mongoclient = pymongo.MongoClient(self.mongo_db_url)

        except Exception as e:
            raise CustomException(e, sys)
        
    def export_collection_as_dataframe(self):
            try:
                database_name = self.data_ingestion_config.database_name
                collection_name = self.data_ingestion_config.collection_name
                collection = self.mongoclient[database_name][collection_name]

                df = pd.DataFrame(list(collection.find()))
                if "_id" in df.columns.to_list():
                    df = df.drop(columns=["_id"], axis =1)
                return df
            
            except Exception as e:
                raise CustomException(e, sys)
        
    def export_data_into_feature_store(self, dataframe: pd.DataFrame) -> pd.DataFrame:
            try:
                feature_store_file_name =self.data_ingestion_config.feature_store_file_name
                create_directory(feature_store_file_name)
                save_data(dataframe, self.data_ingestion_config.feature_store_file_name)
                logging.info(f"saved raw data in to feature store_file_path")
                return dataframe
            except Exception as e:
                raise CustomException(e, sys)
            

    def split_data_as_train_test(self, dataframe:pd.DataFrame):
            try:
                logging.info("spliiting data into train_test_split_ratio")
                train_set, test_set = train_test_split(dataframe , test_size = self.data_ingestion_config.train_test_split_ratio)
                
                #saving training data
                training_file_name = Path(train_set,self.data_ingestion_config.training_file_name)
                create_directory(training_file_name)
                save_data(training_file_name)

                #saving testing data
                testing_file_name = Path(self.data_ingestion_config.testing_file_name)
                create_directory(testing_file_name)
                save_data(test_set, testing_file_name)

            except Exception as e:
                raise CustomException(e, sys)
            

    def initiate_data_ingestion(self) -> DataIngestionArtifact:
            try:
                dataframe = self.export_collection_as_dataframe()
                dataframe = self.export_data_into_feature_store(dataframe)
                self.split_data_as_train_test(dataframe)

                #Returning the artifact
                data_ingestion_artifact = DataIngestionArtifact(
                    training_file_name= Path(self.data_ingestion_config.training_file_name),
                    testing_file_name = Path(self.data_ingestion_config.testing_file_name),
                 )
                return data_ingestion_artifact
            except Exception as e:
                raise CustomException(e, sys)


In [8]:
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.components.data_ingestion import DataIngestion
from src.flight_price_prediction.logging.logger import logger
from src.flight_price_prediction.exception.exception import CustomException
import sys
STAGE_NAME = "Data Ingestion Stage"

class DataIngestionTrainingPipeline:
    def __init__(self, config: ConfigurationManager):
        try:
            self.config = config
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_ingestion(self):
        try:
            data_ingestion_config = self.config.get_data_ingestion_config()
            data_ingestion = DataIngestion(data_ingestion_config)
            data_ingestion_artifact = data_ingestion.initiate_data_ingestion()
            return data_ingestion_artifact
        except Exception as e:
            raise CustomException(e, sys)
        
if __name__ == "__main__":
    try:
        logger.info(f">>>> stage {STAGE_NAME} started <<<<")
        config = ConfigurationManager()
        obj = DataIngestionTrainingPipeline(config =config)
        obj.initiate_data_ingestion()
        logger.info(f">>>> stage {STAGE_NAME} completed <<<<\n\nx====x")
    except Exception as e:
        logger.exception(e)
        raise e


[2026-06-04 03:12:54,916: INFO: 544075182: >>>> stage Data Ingestion Stage started <<<<]
[2026-06-04 03:12:54,923: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/config/config.yaml loaded succesfully ]
[2026-06-04 03:12:54,931: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/params/params.yaml loaded succesfully ]
[2026-06-04 03:12:54,937: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/schema/schema.yaml loaded succesfully ]
[2026-06-04 03:12:54,943: INFO: common: created directory at: artifacts]
[2026-06-04 03:12:54,950: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-04 03:13:02,608: INFO: common: Data saved to: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-04 03:13:02,609: INFO: data_ingestion: saved raw data into feature store file path: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-04 03:13:02,611: INFO: data_ingestion: splitting data into train_test_